# Joan Tryhard

### Imports

In [1]:
import pandas as pd
import sklearn
import imblearn
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix

### Get Data and Preprocess

In [2]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
import pandas as pd

TESTING_WITH_TRAIN_DATA = False
TESTING_WITH_KAGGLE_DATA = True


train = pd.read_csv("data/train_dataset_processed.csv")
test = pd.read_csv("data/test_dataset_processed.csv")

languages = train['language'].unique()


if TESTING_WITH_TRAIN_DATA:

    # Split into train and test sets
    sentences = train[['language', 'sentence_id']].drop_duplicates()
    sentences_train, sentences_test = train_test_split(sentences,test_size=0.2,random_state=42)
    train_set = pd.merge(train, sentences_train, on=['language', 'sentence_id'])
    test_set = pd.merge(train, sentences_test, on=['language', 'sentence_id'])

    # One-hot encode 'language' in train and test sets with language_LANGUAGE
    enc = OneHotEncoder(sparse_output=True)
    language_encoded = enc.fit_transform(train_set[['language']])
    language_df = pd.DataFrame(language_encoded.toarray(),
                            columns=enc.get_feature_names_out(['language']),
                            index=train_set.index)
    train = pd.concat([train_set.drop(columns=['language']), language_df], axis=1)

    # Same for test set
    language_encoded_test = enc.transform(test_set[['language']])
    language_df_test = pd.DataFrame(language_encoded_test.toarray(),
                                    columns=enc.get_feature_names_out(['language']),
                                    index=test_set.index)
    test = pd.concat([test_set.drop(columns=['language']), language_df_test], axis=1)

    # Prepare data and labels
    X_train = train.drop(columns=['root'])
    y_train = train['root']
    X_test = test.drop(columns=['root'])
    y_test = test['root']

else: 
    # One-hot encode 'language' in train and test (all test data) sets
    enc = OneHotEncoder(sparse_output=True)
    language_encoded = enc.fit_transform(train[['language']])
    language_df = pd.DataFrame(language_encoded.toarray(),
                            columns=enc.get_feature_names_out(['language']),
                            index=train.index)
    train = pd.concat([train.drop(columns=['language']), language_df], axis=1)

    # Same for test set
    language_encoded_test = enc.transform(test[['language']])
    language_df_test = pd.DataFrame(language_encoded_test.toarray(),
                                    columns=enc.get_feature_names_out(['language']),
                                    index=test.index)
    test = pd.concat([test.drop(columns=['language']), language_df_test], axis=1)

    #Prepare data and labels
    X_train = train.drop(columns=['root'])
    y_train = train['root']
    X_test = test

In [7]:
X_test

,sentence_id,node,degree,avg_neighbor_deg,degree_squared,degree_diff,clustering,local_degree_ratio,max_neighbor_degree,degree_centrality,harmonic_centrality,betweenness_centrality,pagerank,root,language
0,1,38,2,2.5,4,-0.5,0,0.799997,4,0.047619,8.953882,0.047619,0.024647,0.05,Japanese
1,1,33,1,2.0,1,-1.0,0,0.499998,2,0.023810,7.094756,0.000000,0.013964,0.00,Japanese
2,1,10,4,2.0,16,2.0,0,1.999990,3,0.095238,11.348363,0.335656,0.043718,0.02,Japanese
3,1,24,2,1.5,4,0.5,0,1.333324,2,0.047619,6.855212,0.047619,0.026723,0.00,Japanese
4,1,16,1,2.0,1,-1.0,0,0.499998,2,0.023810,5.649594,0.000000,0.014845,0.01,Japanese
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
194643,993,7,2,3.0,4,-1.0,0,0.666664,4,0.133333,6.266667,0.247619,0.065081,0.01,Russian
194644,993,5,2,1.5,4,0.5,0,1.333324,2,0.133333,5.292857,0.133333,0.070454,0.01,Russian
194645,993,11,1,2.0,1,-1.0,0,0.499998,2,0.066667,4.070238,0.000000,0.039319,0.00,Russian
194646,993,2,2,2.5,4,-0.5,0,0.799997,4,0.133333,6.016667,0.133333,0.067486,0.07,Russian


## Models

### Unimodel Random Forest

In [3]:
from sklearn.ensemble import RandomForestClassifier
# import linear classifier
clf = RandomForestClassifier(class_weight='balanced', random_state = 42)
clf.fit(X_train, y_train)

prob_predictions = clf.predict_proba(X_test)
# convert to 0s and 1s integers
predictions = []
for i in range(len(prob_predictions)):
    if prob_predictions[i][1] > 0.5:
        predictions.append(1)
    else:
        predictions.append(0)

## Multimodel Random Forest

## Evaluating results

In [4]:
if TESTING_WITH_TRAIN_DATA:

    y_test_labels = y_test.values

    # Get the error on the test set with different metrics
    print(confusion_matrix(y_test_labels, predictions))
    print(classification_report(y_test_labels, predictions, target_names=['0', '1']))

In [5]:
def find_root(group):
    # Return the note with greater 'root' value
    return group.loc[group['root'].idxmax()]['node']

    # Check if there's a row with root == 1
    root_rows = group[group['root'] == 1]
    if not root_rows.empty:
        return root_rows.iloc[0]['node']
    else:
        return 1

if not TESTING_WITH_TRAIN_DATA:
    # Add predictions to the test set
    X_test['root'] = prob_predictions[:, 1]  # Use the probability of class 1 as the root value

    # Create a language column with the original language values, which are now one-hot encoded
    language_columns = [col for col in X_test.columns if col.startswith('language_')]
    language_values = enc.inverse_transform(X_test[language_columns])[:, 0]
    X_test['language'] = language_values
    # Drop the one-hot encoded language columns
    X_test = X_test.drop(columns=language_columns)

    # Group by language and sentence_id
    grouped = X_test.groupby(['language', 'sentence_id'])

    # Apply the root finding function to each group
    roots = grouped.apply(find_root).reset_index(name='root')

    # Add a sequential id column
    roots.insert(0, 'id', range(1, len(roots) + 1))
    roots = roots[['id', 'root']]

    # Save to CSV
    roots.to_csv('data/predictions_submission.csv', index=False)

current_predictions = pd.read_csv('data/predictions_submission.csv')
current_predictions.head()

/tmp/ipykernel_156751/2972372113.py:27: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  roots = grouped.apply(find_root).reset_index(name='root')


,id,root
0,1,16
1,2,14
2,3,14
3,4,7
4,5,2


In [6]:
if TESTING_WITH_KAGGLE_DATA:
    kaggle_perfect_predictions = pd.read_csv("data/kaggle_perfect_predictions.csv")
    current_predictions = pd.read_csv('data/predictions_submission.csv')
    kaggle_perfect_predictions = kaggle_perfect_predictions.drop(columns=['id'])
    current_predictions = current_predictions.drop(columns=['id'])
    current_predictions = current_predictions.drop(index=0)
    kaggle_perfect_predictions = kaggle_perfect_predictions.drop(index=0)
    # Get the error on the kaggle test set 

print(classification_report(kaggle_perfect_predictions, current_predictions))

              precision    recall  f1-score   support

           1       0.09      0.08      0.09       690
           2       0.09      0.08      0.08       675
           3       0.12      0.12      0.12       687
           4       0.10      0.11      0.11       640
           5       0.11      0.12      0.12       693
           6       0.10      0.10      0.10       626
           7       0.09      0.09      0.09       653
           8       0.08      0.08      0.08       607
           9       0.10      0.10      0.10       547
          10       0.08      0.08      0.08       510
          11       0.07      0.08      0.08       491
          12       0.09      0.10      0.10       407
          13       0.07      0.07      0.07       390
          14       0.06      0.06      0.06       346
          15       0.07      0.07      0.07       336
          16       0.06      0.06      0.06       312
          17       0.05      0.05      0.05       248
          18       0.06    

/home/joan/Master-Data-Science-FIB-UPC/2nd semester/ML- Machine Learning/ML-Machine-Learning/PROJECT/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/joan/Master-Data-Science-FIB-UPC/2nd semester/ML- Machine Learning/ML-Machine-Learning/PROJECT/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/joan/Master-Data-Science-FIB-UPC/2nd semester/ML- Machine Learning/ML-Machine-Learning/PROJECT/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: Undefine